In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import mutual_info_regression, RFE
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


In [ ]:
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

df = pd.read_csv('laptop_price_finale.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
corr_with_target = X.corrwith(y).sort_values(key=np.abs, ascending=False)
corr_df = corr_with_target.to_frame('pearson_corr')
corr_df['abs_corr'] = corr_df['pearson_corr'].abs()
corr_df.head(15)

In [ ]:
top_n = 15
top_corr = corr_df.head(top_n).sort_values('abs_corr')

plt.figure(figsize=(8, 6))
colors = ['#d62728' if v < 0 else '#1f77b4' for v in top_corr['pearson_corr']]
plt.barh(top_corr.index, top_corr['pearson_corr'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title(f'Top {top_n} features by |Pearson correlation| with Price_log')
plt.xlabel('Pearson correlation')
plt.tight_layout()
plt.savefig('fig_corr_filter.png')
plt.show()

In [ ]:
FILTER_CORR_SET = set(corr_df.head(top_n).index)
FILTER_CORR_SET

In [ ]:
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
mi_df = mi_series.to_frame('mutual_info')
mi_df.head(15)

In [ ]:
top_mi = mi_series.head(top_n).sort_values()

plt.figure(figsize=(8, 6))
plt.barh(top_mi.index, top_mi.values, color='#2ca02c')
plt.title(f'Top {top_n} features by Mutual Information with Price_log')
plt.xlabel('Mutual information score')
plt.tight_layout()
plt.savefig('fig_mi_filter.png')
plt.show()

In [ ]:
FILTER_MI_SET = set(mi_series.head(top_n).index)
FILTER_MI_SET

In [ ]:
lasso = Lasso(alpha=0.005, random_state=42, max_iter=10000)
lasso.fit(X_train, y_train)

lasso_coef = pd.Series(lasso.coef_, index=X.columns)
lasso_nonzero = lasso_coef[lasso_coef != 0].sort_values(key=np.abs, ascending=False)

print(f'{(lasso_coef != 0).sum()} / {len(lasso_coef)} features kept a non-zero coefficient')
lasso_nonzero.to_frame('lasso_coef').head(15)

In [ ]:
top_lasso = lasso_nonzero.head(top_n).sort_values()

plt.figure(figsize=(8, 6))
colors = ['#d62728' if v < 0 else '#1f77b4' for v in top_lasso.values]
plt.barh(top_lasso.index, top_lasso.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title(f'Top {min(top_n, len(lasso_nonzero))} features by |Lasso coefficient|')
plt.xlabel('Lasso coefficient (on scaled features)')
plt.tight_layout()
plt.savefig('fig_lasso_embedded.png')
plt.show()

In [ ]:
EMBEDDED_LASSO_SET = set(lasso_nonzero.head(top_n).index)
EMBEDDED_LASSO_SET

In [ ]:
rf = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
rf_importance.head(15).to_frame('rf_importance')

In [ ]:
top_rf = rf_importance.head(top_n).sort_values()

plt.figure(figsize=(8, 6))
plt.barh(top_rf.index, top_rf.values, color='#9467bd')
plt.title(f'Top {top_n} features by Random Forest importance')
plt.xlabel('Importance (mean decrease in impurity)')
plt.tight_layout()
plt.savefig('fig_rf_embedded.png')
plt.show()

In [ ]:
EMBEDDED_RF_SET = set(rf_importance.head(top_n).index)
EMBEDDED_RF_SET

In [ ]:
estimator = LinearRegression()
rfe = RFE(estimator, n_features_to_select=top_n, step=1)
rfe.fit(X_train, y_train)

rfe_ranking = pd.Series(rfe.ranking_, index=X.columns).sort_values()
rfe_selected = rfe_ranking[rfe_ranking == 1]
print(f'{len(rfe_selected)} features selected directly by RFE (rank = 1)')
rfe_ranking.head(20).to_frame('rfe_rank')

In [ ]:
plt.figure(figsize=(8, 6))
plot_ranks = rfe_ranking.head(top_n).sort_values(ascending=False)
plt.barh(plot_ranks.index, plot_ranks.values, color='#ff7f0e')
plt.title(f'RFE rank of top {top_n} features (1 = selected)')
plt.xlabel('RFE ranking (lower = more important)')
plt.tight_layout()
plt.savefig('fig_rfe_wrapper.png')
plt.show()

In [ ]:
WRAPPER_RFE_SET = set(rfe_selected.index)
WRAPPER_RFE_SET

In [ ]:
all_features = sorted(set(X.columns))
method_sets = {
    'Pearson Corr.': FILTER_CORR_SET,
    'Mutual Info': FILTER_MI_SET,
    'Lasso': EMBEDDED_LASSO_SET,
    'Random Forest': EMBEDDED_RF_SET,
    'RFE': WRAPPER_RFE_SET,
}

selection_matrix = pd.DataFrame(
    {name: [feat in feat_set for feat in all_features] for name, feat_set in method_sets.items()},
    index=all_features
)
selection_matrix['vote_count'] = selection_matrix.sum(axis=1)
selection_matrix = selection_matrix.sort_values('vote_count', ascending=False)
selection_matrix.head(20)